<a href="https://colab.research.google.com/github/B-MEbrahim/ECHO-Tasks/blob/main/Week1/Echo_Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence_transformers langchain langchain-text-splitters langchain-nvidia-ai-endpoints

In [ ]:
import nltk

nltk.download("punkt_tab")
nltk.download("punkt")

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

# Download from the public tmdb-5000 repo
file_path = hf_hub_download(
    repo_id="alejandrowallace/tmdb-5000",
    filename="tmdb_5000_movies.csv",
    repo_type="dataset",
)

# Load into pandas and filter
df = pd.read_csv(file_path)
df = df[["original_title", "overview"]].dropna().reset_index(drop=True)

# Save locally as movies.csv for downstream tutorial steps
df.to_csv("movies.csv", index=False)

print(f"Loaded successfully! Shape: {df.shape}")
print(df.head())

## Original Code

In [19]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import NLTKTextSplitter

df = df[['original_title', 'overview']]

# Step 1: Chunk the Overview Column using NLTKTextSplitter
text_splitter = NLTKTextSplitter(chunk_size=1500)

def split_overview(overview):
    if pd.isna(overview):
        return []
    return text_splitter.split_text(str(overview))

df['chunks'] = df['overview'].apply(split_overview)

# Flatten the dataframe for easier processing
chunked_df = df.explode('chunks').reset_index(drop=True)

# Step 2: Embed with a SentenceTransformer Encoder
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # You can choose another model

# Make sure all chunks are strings and not empty
def encode_chunk(chunk):
    if not isinstance(chunk, str) or chunk.strip() == "":
        return None
    return embedder.encode(chunk).tolist()

chunked_df['embeddings'] = chunked_df['chunks'].apply(encode_chunk)

# Drop rows where 'embeddings' is None
chunked_df.dropna(subset=['embeddings'], inplace=True)

# Step 3: Store in ChromaDB
# Initialize ChromaDB client and collection
client = chromadb.Client()
collection = client.create_collection(name='movies')

# Insert data into ChromaDB
for idx, row in chunked_df.iterrows():
    collection.add(
        ids=[str(idx)],
        embeddings=[row['embeddings']],
        metadatas=[{
            'original_title': row['original_title'],
            'chunk': row['chunks']
        }]
    )

print("Data successfully stored in ChromaDB.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Data successfully stored in ChromaDB.


In [ ]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from chromadb import Client
from sentence_transformers import SentenceTransformer
import chromadb
import torch

# Load the SentenceTransformer model for encoding queries
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  # Use the same model as for embedding documents

# Load the text generation model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.1",
    device_map='auto' # load it in the current GPU
)
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    return_full_text=True,
    max_new_tokens=800
)


# Function to retrieve top_k documents from ChromaDB
def retrieve_documents(query, collection, top_k=5):
    # Embed the query using the SentenceTransformer model
    query_embedding = sentence_model.encode(query).tolist()

    # Search for top_k similar documents in the collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    if not results['documents']:
        print("No results found for the query.")
        return [], []

    # Extract chunks and titles from the results
    chunks = []
    titles = []
    for document in results['metadatas'][0]:
        chunks.append(document['chunk'])
        titles.append(document['original_title'])

    return chunks, titles

# Function to generate answer based on retrieved chunks and titles
def generate_answer(query, chunks, titles, text_generation_pipeline):
    # Prepare the context from chunks and titles
    context = "\n\n".join([f"Title: {title}\nChunk: {chunk}" for title, chunk in zip(titles, chunks)])

    # Prepare the prompt
    prompt = f"""[INST]
    Instruction: You're an expert in movie suggestions. Your task is to analyze carefully the context and come up with an exhaustive answer to the following question:
    {query}

    Here is the context to help you:

    {context}

    [/INST]"""

    # Generate the answer using the model
    generated_text = text_generation_pipeline(prompt)[0]['generated_text']

    return generated_text

In [31]:
# Example usage
client = chromadb.Client()
collection = client.get_collection(name='movies')

query = """Choose life, choose a job, choose a career, choose a big television...
movie about Edinburgh drug culture."""
top_k = 3

# Retrieve documents
chunks, titles = retrieve_documents(query, collection, top_k)
print(f"Query: {query}\n")
print("="*80)
print(f"TOP {top_k} RETRIEVED RESULTS:")
print("="*80)
for idx, (title, chunk) in enumerate(zip(titles, chunks), 1):
    print(f"\n[Result #{idx}] Title: {title}")
    print("-" * 40)
    print(chunk)
    print("-" * 80)

Query: Choose life, choose a job, choose a career, choose a big television... 
movie about Edinburgh drug culture.

TOP 3 RETRIEVED RESULTS:

[Result #1] Title: Trainspotting
----------------------------------------
Renton, deeply immersed in the Edinburgh drug scene, tries to clean up and get out, despite the allure of the drugs and influence of friends.
--------------------------------------------------------------------------------

[Result #2] Title: Get Rich or Die Tryin'
----------------------------------------
A tale of an inner city drug dealer who turns away from crime to pursue his passion, rap music.
--------------------------------------------------------------------------------

[Result #3] Title: London
----------------------------------------
London is a drug laden adventure that centers on a party in a New York loft where a young man is trying to win back his ex-girlfriend.
--------------------------------------------------------------------------------


## Modified Code

In [21]:
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import CharacterTextSplitter


df = df[['original_title', 'overview']]

# Step 1: Chunk the Overview Column using NLTKTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=1500,
                                 chunk_overlap=20)

def split_overview(overview):
    if pd.isna(overview):
        return []
    return text_splitter.split_text(str(overview))

df['chunks'] = df['overview'].apply(split_overview)

# Flatten the dataframe for easier processing
chunked_df = df.explode('chunks').reset_index(drop=True)

# Step 2: Embed with a SentenceTransformer Encoder
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # You can choose another model

# Make sure all chunks are strings and not empty
def encode_chunk(chunk):
    if not isinstance(chunk, str) or chunk.strip() == "":
        return None
    return embedder.encode(chunk).tolist()

chunked_df['embeddings'] = chunked_df['chunks'].apply(encode_chunk)

# Drop rows where 'embeddings' is None
chunked_df.dropna(subset=['embeddings'], inplace=True)

# Step 3: Store in ChromaDB
# Initialize ChromaDB client and collection
client = chromadb.Client()
collection = client.create_collection(name='movies2')

# Insert data into ChromaDB
for idx, row in chunked_df.iterrows():
    collection.add(
        ids=[str(idx)],
        embeddings=[row['embeddings']],
        metadatas=[{
            'original_title': row['original_title'],
            'chunk': row['chunks']
        }]
    )

print("Data successfully stored in ChromaDB.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Data successfully stored in ChromaDB.


In [30]:
# Example usage
client = chromadb.Client()
collection = client.get_collection(name='movies2')

query = """Choose life, choose a job, choose a career, choose a big television...
movie about Edinburgh drug culture."""
top_k = 3

# Retrieve documents
chunks, titles = retrieve_documents(query, collection, top_k)


print(f"Query: {query}\n")
print("="*80)
print(f"TOP {top_k} RETRIEVED RESULTS:")
print("="*80)
for idx, (title, chunk) in enumerate(zip(titles, chunks), 1):
    print(f"\n[Result #{idx}] Title: {title}")
    print("-" * 40)
    print(chunk)
    print("-" * 80)

Query: Choose life, choose a job, choose a career, choose a big television... 
movie about Edinburgh drug culture.

TOP 3 RETRIEVED RESULTS:

[Result #1] Title: Trainspotting
----------------------------------------
Renton, deeply immersed in the Edinburgh drug scene, tries to clean up and get out, despite the allure of the drugs and influence of friends.
--------------------------------------------------------------------------------

[Result #2] Title: Get Rich or Die Tryin'
----------------------------------------
A tale of an inner city drug dealer who turns away from crime to pursue his passion, rap music.
--------------------------------------------------------------------------------

[Result #3] Title: London
----------------------------------------
London is a drug laden adventure that centers on a party in a New York loft where a young man is trying to win back his ex-girlfriend.
--------------------------------------------------------------------------------


---

## Observations
In the modfied code i Used `CharacterTextSplitter` with an overlapping paramter of `20` and created another collection called `movies2`.

Both chunking methods seem to generate the same chunks but the way they're generated is different.

*Both method yielded the movie i was looking for which is `Trainspotting`*


![](https://i.pinimg.com/736x/88/89/fa/8889fa22ca61bd7b07ff57fc6bf55f28.jpg)

## Apologies For The Late Submission!
![](https://i.pinimg.com/736x/80/b7/e2/80b7e23918d1ce91b741e0a2816c3ae6.jpg)